In [0]:
#1. Ingest sample order data into a Spark DataFrame.
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_timestamp", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True)
])

countries = ["US","IN","GB","DE","FR","BR","CA"]
currencies = {"US":"USD","IN":"INR","GB":"GBP","DE":"EUR","FR":"EUR","BR":"BRL","CA":"CAD"}
statuses = ["CREATED", "PAID", "CANCELLED"]

base_time = datetime.utcnow()

data_rows = []
for i in range(10000):
    timestamp = base_time - timedelta(
        days=random.randint(0, 5),
        hours=random.randint(0, 23)
    )
    country = random.choice(countries)
    amount = round(random.uniform(10, 500), 2)  # Use Python's round
    data_rows.append((
        f"ORD{i:04d}",
        timestamp,
        f"CUST{random.randint(1, 50):03d}",
        country,
        amount,
        currencies[country],
        random.choice(statuses)
    ))

orders_df = spark.createDataFrame(data_rows, schema=order_schema)
display(orders_df)

/home/spark-5bb82339-72dd-4c8e-92de-be/.ipykernel/4457/command-6427955251613166-845419916:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time = datetime.utcnow()


order_id,order_timestamp,customer_id,country,amount,currency,status
ORD0000,2025-11-26T22:01:41.929Z,CUST022,IN,469.35,INR,CANCELLED
ORD0001,2025-11-28T00:01:41.929Z,CUST015,US,223.11,USD,CREATED
ORD0002,2025-12-01T12:01:41.929Z,CUST006,BR,50.32,BRL,PAID
ORD0003,2025-11-26T05:01:41.929Z,CUST043,BR,112.51,BRL,PAID
ORD0004,2025-11-28T01:01:41.929Z,CUST010,DE,435.39,EUR,PAID
ORD0005,2025-11-30T05:01:41.929Z,CUST036,CA,203.58,CAD,CANCELLED
ORD0006,2025-11-30T12:01:41.929Z,CUST028,GB,454.97,GBP,CANCELLED
ORD0007,2025-11-29T08:01:41.929Z,CUST007,CA,109.84,CAD,CANCELLED
ORD0008,2025-11-28T11:01:41.929Z,CUST018,IN,221.86,INR,CREATED
ORD0009,2025-11-30T05:01:41.929Z,CUST035,FR,400.36,EUR,CREATED


In [0]:
#2.Add a derived column order_date (date only from order_timestamp).
from pyspark.sql.functions import to_date, col

orders_df_with_date = orders_df.withColumn(
    "order_date",
    to_date(col("order_timestamp"))
)

display(orders_df_with_date)


order_id,order_timestamp,customer_id,country,amount,currency,status,order_date
ORD0000,2025-11-26T22:01:41.929Z,CUST022,IN,469.35,INR,CANCELLED,2025-11-26
ORD0001,2025-11-28T00:01:41.929Z,CUST015,US,223.11,USD,CREATED,2025-11-28
ORD0002,2025-12-01T12:01:41.929Z,CUST006,BR,50.32,BRL,PAID,2025-12-01
ORD0003,2025-11-26T05:01:41.929Z,CUST043,BR,112.51,BRL,PAID,2025-11-26
ORD0004,2025-11-28T01:01:41.929Z,CUST010,DE,435.39,EUR,PAID,2025-11-28
ORD0005,2025-11-30T05:01:41.929Z,CUST036,CA,203.58,CAD,CANCELLED,2025-11-30
ORD0006,2025-11-30T12:01:41.929Z,CUST028,GB,454.97,GBP,CANCELLED,2025-11-30
ORD0007,2025-11-29T08:01:41.929Z,CUST007,CA,109.84,CAD,CANCELLED,2025-11-29
ORD0008,2025-11-28T11:01:41.929Z,CUST018,IN,221.86,INR,CREATED,2025-11-28
ORD0009,2025-11-30T05:01:41.929Z,CUST035,FR,400.36,EUR,CREATED,2025-11-30


In [0]:
#3. Write the DataFrame as a Delta table partitioned by country and order_date.
volume_path = "/Volumes/krishna/default/ordervol"

(orders_df_with_date
 .write
 .format("delta")
 .mode("overwrite")
 .partitionBy("country", "order_date")
 .save(volume_path)) 

In [0]:
#4.Verify the partition structure in the storage path.
display(dbutils.fs.ls("/Volumes/krishna/default/ordervol"))


path,name,size,modificationTime
dbfs:/Volumes/krishna/default/ordervol/_delta_log/,_delta_log/,0,1764611725000
dbfs:/Volumes/krishna/default/ordervol/country=BR/,country=BR/,0,1764612114000
dbfs:/Volumes/krishna/default/ordervol/country=CA/,country=CA/,0,1764612114000
dbfs:/Volumes/krishna/default/ordervol/country=DE/,country=DE/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=FR/,country=FR/,0,1764612114000
dbfs:/Volumes/krishna/default/ordervol/country=GB/,country=GB/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=IN/,country=IN/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=US/,country=US/,0,1764611727000


In [0]:
#Displays the structure in orders_df_with_date with tha country partitioning
display(dbutils.fs.ls("/Volumes/krishna/default/ordervol/country=US"))

path,name,size,modificationTime
dbfs:/Volumes/krishna/default/ordervol/country=US/order_date=2025-11-25/,order_date=2025-11-25/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=US/order_date=2025-11-26/,order_date=2025-11-26/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=US/order_date=2025-11-27/,order_date=2025-11-27/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=US/order_date=2025-11-28/,order_date=2025-11-28/,0,1764611728000
dbfs:/Volumes/krishna/default/ordervol/country=US/order_date=2025-11-29/,order_date=2025-11-29/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=US/order_date=2025-11-30/,order_date=2025-11-30/,0,1764611727000
dbfs:/Volumes/krishna/default/ordervol/country=US/order_date=2025-12-01/,order_date=2025-12-01/,0,1764611728000


In [0]:
#5.Run queries that demonstrate partition pruning (e.g., filter on a single country and/or date).
#filter by a single country
df_prune_country = spark.read.format("delta").load(volume_path) \
    .filter(col("country") == "US")
df_prune_country.explain(True)   
display(df_prune_country.limit(10))
#filter by country and order_date
df_prune_country_date = spark.read.format("delta").load(volume_path) \
    .filter(
        (col("country") == "US") &
        (col("order_date") == "2025-01-10")      # replace with actual date seen in your partitions
    )
df_prune_country_date.explain(True)
display(df_prune_country_date.limit(10))
#query Without filters 
df_full = spark.read.format("delta").load(volume_path)
df_full.explain(True)

== Parsed Logical Plan ==
'Filter '`==`('country, US)
+- Relation [order_id#12053,order_timestamp#12054,customer_id#12055,country#12056,amount#12057,currency#12058,status#12059,order_date#12060] parquet

== Analyzed Logical Plan ==
order_id: string, order_timestamp: timestamp, customer_id: string, country: string, amount: double, currency: string, status: string, order_date: date
Filter (country#12056 = US)
+- Relation [order_id#12053,order_timestamp#12054,customer_id#12055,country#12056,amount#12057,currency#12058,status#12059,order_date#12060] parquet

== Optimized Logical Plan ==
Filter (isnotnull(country#12056) AND (country#12056 = US))
+- Relation [order_id#12053,order_timestamp#12054,customer_id#12055,country#12056,amount#12057,currency#12058,status#12059,order_date#12060] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- PhotonResultStage
   +- PhotonProject [order_id#12053, order_timestamp#12054, customer_id#12055, country#12056, amount#12057, currency#12058, status#12059, ord

order_id,order_timestamp,customer_id,country,amount,currency,status,order_date
ORD0066,2025-12-01T09:01:41.929Z,CUST004,US,220.36,USD,PAID,2025-12-01
ORD0098,2025-12-01T02:01:41.929Z,CUST006,US,407.97,USD,CANCELLED,2025-12-01
ORD0164,2025-12-01T06:01:41.929Z,CUST041,US,334.82,USD,CREATED,2025-12-01
ORD0184,2025-12-01T10:01:41.929Z,CUST006,US,279.17,USD,CANCELLED,2025-12-01
ORD0211,2025-12-01T04:01:41.929Z,CUST041,US,190.9,USD,PAID,2025-12-01
ORD0228,2025-12-01T00:01:41.929Z,CUST021,US,266.8,USD,CANCELLED,2025-12-01
ORD0302,2025-12-01T01:01:41.929Z,CUST033,US,253.41,USD,PAID,2025-12-01
ORD0569,2025-12-01T04:01:41.929Z,CUST004,US,436.3,USD,CANCELLED,2025-12-01
ORD0582,2025-12-01T06:01:41.929Z,CUST013,US,99.96,USD,CANCELLED,2025-12-01
ORD0586,2025-12-01T18:01:41.929Z,CUST032,US,120.56,USD,PAID,2025-12-01


== Parsed Logical Plan ==
'Filter 'and('`==`('country, US), '`==`('order_date, 2025-01-10))
+- Relation [order_id#12110,order_timestamp#12111,customer_id#12112,country#12113,amount#12114,currency#12115,status#12116,order_date#12117] parquet

== Analyzed Logical Plan ==
order_id: string, order_timestamp: timestamp, customer_id: string, country: string, amount: double, currency: string, status: string, order_date: date
Filter ((country#12113 = US) AND (order_date#12117 = cast(2025-01-10 as date)))
+- Relation [order_id#12110,order_timestamp#12111,customer_id#12112,country#12113,amount#12114,currency#12115,status#12116,order_date#12117] parquet

== Optimized Logical Plan ==
LocalRelation <empty>, [order_id#12110, order_timestamp#12111, customer_id#12112, country#12113, amount#12114, currency#12115, status#12116, order_date#12117]

== Physical Plan ==
LocalTableScan <empty>, [order_id#12110, order_timestamp#12111, customer_id#12112, country#12113, amount#12114, currency#12115, status#12116

order_id,order_timestamp,customer_id,country,amount,currency,status,order_date


== Parsed Logical Plan ==
Relation [order_id#12159,order_timestamp#12160,customer_id#12161,country#12162,amount#12163,currency#12164,status#12165,order_date#12166] parquet

== Analyzed Logical Plan ==
order_id: string, order_timestamp: timestamp, customer_id: string, country: string, amount: double, currency: string, status: string, order_date: date
Relation [order_id#12159,order_timestamp#12160,customer_id#12161,country#12162,amount#12163,currency#12164,status#12165,order_date#12166] parquet

== Optimized Logical Plan ==
Relation [order_id#12159,order_timestamp#12160,customer_id#12161,country#12162,amount#12163,currency#12164,status#12165,order_date#12166] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- PhotonResultStage
   +- PhotonProject [order_id#12159, order_timestamp#12160, customer_id#12161, country#12162, amount#12163, currency#12164, status#12165, order_date#12166]
      +- PhotonScan parquet [order_id#12159,order_timestamp#12160,customer_id#12161,amount#12163,currency#121

In [0]:
#6.Demonstrate Delta Lake Time Travel: Write data, update some rows, then query older versions.
delta_path = "/Volumes/krishna/default/ordervol"

(orders_df_with_date
 .write
 .format("delta")
 .mode("overwrite")
 .partitionBy("country", "order_date")
 .save(delta_path))
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit
delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.update(
    condition = col("amount") < 50,
    set = { "status": lit("CANCELLED") }
)
#query oldr version using time travel
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path)
#query latest version 
display(df_old.filter(col("amount") < 50)) 
df_latest = spark.read.format("delta").load(delta_path)
display(df_latest.filter(col("amount") < 50))  

order_id,order_timestamp,customer_id,country,amount,currency,status,order_date
ORD0010,2025-11-26T03:49:26.145Z,CUST009,GB,31.11,GBP,PAID,2025-11-26
ORD0013,2025-11-29T04:49:26.145Z,CUST018,DE,29.45,EUR,CANCELLED,2025-11-29
ORD0079,2025-11-29T19:49:26.145Z,CUST029,DE,28.31,EUR,CANCELLED,2025-11-29
ORD0053,2025-11-29T10:49:26.145Z,CUST020,US,38.38,USD,CREATED,2025-11-29
ORD0048,2025-12-01T14:49:26.145Z,CUST028,DE,27.95,EUR,CANCELLED,2025-12-01


order_id,order_timestamp,customer_id,country,amount,currency,status,order_date
ORD0547,2025-11-28T06:01:41.929Z,CUST024,BR,49.03,BRL,CANCELLED,2025-11-28
ORD0613,2025-11-28T10:01:41.929Z,CUST031,BR,37.33,BRL,CANCELLED,2025-11-28
ORD0811,2025-11-28T11:01:41.929Z,CUST031,BR,46.81,BRL,CANCELLED,2025-11-28
ORD1071,2025-11-28T22:01:41.929Z,CUST020,BR,36.24,BRL,CANCELLED,2025-11-28
ORD1119,2025-11-28T00:01:41.929Z,CUST034,BR,29.83,BRL,CANCELLED,2025-11-28
ORD1566,2025-11-28T06:01:41.929Z,CUST037,BR,18.7,BRL,CANCELLED,2025-11-28
ORD1646,2025-11-28T19:01:41.929Z,CUST041,BR,23.01,BRL,CANCELLED,2025-11-28
ORD1749,2025-11-28T12:01:41.929Z,CUST024,BR,39.95,BRL,CANCELLED,2025-11-28
ORD3853,2025-11-28T03:01:41.929Z,CUST034,BR,28.12,BRL,CANCELLED,2025-11-28
ORD4031,2025-11-28T16:01:41.929Z,CUST017,BR,23.88,BRL,CANCELLED,2025-11-28


In [0]:
#7. Demonstrate Schema Evolution: Add payment_method & coupon_code to new data. Write to the same Delta table, allowing schema evolution.
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import to_date, col
import uuid, random, datetime

# New batch size
N_NEW = 500

payment_methods = ["CARD", "UPI", "COD", "WALLET"]

new_rows = []
for _ in range(N_NEW):
    oid = str(uuid.uuid4())
    ts = rand_ts(START, END + datetime.timedelta(days=10))
    cid = f"{random.randint(1,4000)}"
    country = random.choice(COUNTRIES)
    amount = round(random.uniform(1.0,500.0),2)
    currency = CURRENCIES[country]
    status = random.choice(STATUSES)
    payment_method = random.choice(payment_methods)
    coupon_code = random.choice([None, f"CPN{random.randint(100,999)}", None])

    new_rows.append((oid, ts, cid, country, amount, currency, status, payment_method, coupon_code))

# Schema with new fields
schema_new = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_timestamp", TimestampType(), False),
    StructField("customer_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("status", StringType(), False),
    StructField("payment_method", StringType(), True),
    StructField("coupon_code", StringType(), True)
])

df_new = spark.createDataFrame(new_rows, schema_new)
df_new = df_new.withColumn("order_date", to_date(col("order_timestamp")))

display(df_new.limit(5))


order_id,order_timestamp,customer_id,country,amount,currency,status,payment_method,coupon_code,order_date
35bae3c9-9c7e-4a50-8c23-603f24484293,2025-01-21T07:29:20.000Z,161,UK,469.19,GBP,SHIPPED,WALLET,CPN987,2025-01-21
fb1a7bc5-fc69-4c4c-8e77-dbcdd5f3bb4f,2025-01-28T22:59:34.000Z,3342,DE,169.5,EUR,DELIVERED,WALLET,CPN130,2025-01-28
63762159-0cf8-42b4-8ea5-ad38542958e8,2025-02-03T03:11:51.000Z,1573,IN,308.05,INR,DELIVERED,CARD,CPN252,2025-02-03
ebe781b9-9c1b-41cb-ba1c-54d72b9c21c0,2025-01-20T20:03:14.000Z,3126,US,96.26,USD,PLACED,CARD,null,2025-01-20
c85776be-256b-413a-9418-da14650a0216,2025-01-30T19:25:41.000Z,1598,DE,91.61,EUR,SHIPPED,COD,CPN690,2025-01-30


In [0]:
'''Demonstrate Updates &amp; Deletes using Delta:
Mark some orders as CANCELLED.
Delete orders below a certain amount (e.g., test data cleanup).'''
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit
delta_table = DeltaTable.forPath(spark, delta_path)

delta_table.update(
    condition = col("amount") < 50,
    set = {"status": lit("CANCELLED")}
)

#verifying the updates
display(spark.read.format("delta").load(delta_path).filter(col("amount") < 50))
delta_table.delete(
    condition = col("amount") < 10
)
#verifying the deletes
display(spark.read.format("delta").load(delta_path).filter(col("amount") < 10))


order_id,order_timestamp,customer_id,country,amount,currency,status,order_date
ORD0547,2025-11-28T06:01:41.929Z,CUST024,BR,49.03,BRL,CANCELLED,2025-11-28
ORD0613,2025-11-28T10:01:41.929Z,CUST031,BR,37.33,BRL,CANCELLED,2025-11-28
ORD0811,2025-11-28T11:01:41.929Z,CUST031,BR,46.81,BRL,CANCELLED,2025-11-28
ORD1071,2025-11-28T22:01:41.929Z,CUST020,BR,36.24,BRL,CANCELLED,2025-11-28
ORD1119,2025-11-28T00:01:41.929Z,CUST034,BR,29.83,BRL,CANCELLED,2025-11-28
ORD1566,2025-11-28T06:01:41.929Z,CUST037,BR,18.7,BRL,CANCELLED,2025-11-28
ORD1646,2025-11-28T19:01:41.929Z,CUST041,BR,23.01,BRL,CANCELLED,2025-11-28
ORD1749,2025-11-28T12:01:41.929Z,CUST024,BR,39.95,BRL,CANCELLED,2025-11-28
ORD3853,2025-11-28T03:01:41.929Z,CUST034,BR,28.12,BRL,CANCELLED,2025-11-28
ORD4031,2025-11-28T16:01:41.929Z,CUST017,BR,23.88,BRL,CANCELLED,2025-11-28


order_id,order_timestamp,customer_id,country,amount,currency,status,order_date


In [0]:
'''9. (Bonus) Optimize the table:
Use OPTIMIZE and optionally ZORDER on customer_id or order_date.'''
delta_table = DeltaTable.forPath(spark, delta_path)
spark.sql(f"OPTIMIZE delta.`{delta_path}`")
spark.sql(f"OPTIMIZE delta.`{delta_path}` ZORDER BY (customer_id)")


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
'''10. (Bonus) Show how small file problems can occur with too many partitions and how
OPTIMIZE helps.'''
import pyspark.sql.functions as F

#list files in delta table storage path
files = dbutils.fs.ls(delta_path)

print("Files in Delta table root:")
for f in files:
    print(f.name)

#count number of files per country 
spark.read.format("delta").load(delta_path)\
    .groupBy("country")\
    .agg(F.count("*").alias("num_rows"))\
    .show()
import time

start = time.time()
spark.read.format("delta").load(delta_path)\
    .filter(col("customer_id") == "100")\
    .show()
end = time.time()
print(f"Query took {end-start:.2f} seconds with small files")
# Basic OPTIMIZE to compact small files
spark.sql(f"OPTIMIZE delta.`{delta_path}`")
#basic OPTIMIZE to compact small files
spark.sql(f"OPTIMIZE delta.`{delta_path}`")
#now only on non-partition columns 
spark.sql(f"OPTIMIZE delta.`{delta_path}` ZORDER BY (customer_id)")


Files in Delta table root:
_delta_log/
country=BR/
country=CA/
country=DE/
country=FR/
country=GB/
country=IN/
country=US/
deletion_vector_26a6882b-8bbc-4e82-84a5-0ed8c691253a.bin
deletion_vector_274fc4da-95f7-4222-bd76-77924de25135.bin
+-------+--------+
|country|num_rows|
+-------+--------+
|     GB|    1453|
|     US|    1385|
|     BR|    1458|
|     DE|    1352|
|     CA|    1438|
|     FR|    1461|
|     IN|    1453|
+-------+--------+

+--------+---------------+-----------+-------+------+--------+------+----------+
|order_id|order_timestamp|customer_id|country|amount|currency|status|order_date|
+--------+---------------+-----------+-------+------+--------+------+----------+
+--------+---------------+-----------+-------+------+--------+------+----------+

Query took 0.24 seconds with small files


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,